# script 1: INGESTA DE BRONZEEE DONDE COGEMOS LOS DATOS Y LOS TRANSLADAMOS DE FUENTEEEE (http-to-bucket)
en este script bajamos el archivo parquet de los taxis de nyc de enero 2025 directamente desde la url publica y lo guardamos crudo en el minion en el bucket `taxis` (capa bronze) usando dlt.

> **nota:** NO SE USA PIP INSTALL, ESO ESTÁ EN REQUIREMENTS

In [1]:
import dlt
import pyarrow.parquet as pq
import fsspec

# 1. definimos el recurso dlt que lee los taxis desde la url publica
@dlt.resource(table_name="df_data")
def my_df():
    parquet_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
    print(f"descargando el parquet: {parquet_url}")
    with fsspec.open(parquet_url, mode="rb") as f:
        table = pq.read_table(f)
        df = table.to_pandas()
        print(f"total de registros leidos: {len(df):,}")
        yield df

# 2.PIPELINE PARA TRANSLADAR LOS TAXIS AL MINION
# las credenciales las saca automatico de .dlt/secrets.toml
pipeline = dlt.pipeline(
    pipeline_name="parquet_to_minio",
    destination="filesystem",
    dataset_name="taxis_parquet",
)

# 3. corremos la carga al minion
load_info = pipeline.run(
    my_df,
    loader_file_format="parquet",
    write_disposition="replace"
)

print("carga a minion")
print(load_info)

2026-09-06 14:46:00,394|[INFO]|1593|126284444890944|dlt|pipeline.py|_restore_state_from_destination:1947|The state was restored from the destination filesystem (dlt.destinations.filesystem):taxis_parquet


descargando el parquet: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet


/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


total de registros leidos: 3,475,226


2026-09-06 14:46:45,780|[INFO]|1593|126284444890944|dlt|pool_runner.py|create_pool:203|Created none pool with 1 workers
2026-09-06 14:46:45,815|[INFO]|1593|126284444890944|dlt|normalize.py|run:309|Running file normalizing
2026-09-06 14:46:45,832|[INFO]|1593|126284444890944|dlt|normalize.py|run:312|Found 1 load packages
2026-09-06 14:46:45,882|[INFO]|1593|126284444890944|dlt|normalize.py|run:335|Found 1 files in schema parquet_to_minio load_id 1788705960.7305224
2026-09-06 14:46:46,230|[INFO]|1593|126284444890944|dlt|normalize.py|spool_schema_files:298|Created new load package 1788705960.7305224 on loading volume with 1 files
2026-09-06 14:46:46,332|[INFO]|1593|126284444890944|dlt|worker.py|_get_items_normalizer:137|A file format for table df_data was specified to parquet in the resource so parquet format being used.
2026-09-06 14:46:46,334|[INFO]|1593|126284444890944|dlt|worker.py|_get_items_normalizer:186|Created items normalizer ArrowItemsNormalizer with writer ArrowToParquetWriter f

carga a minion
Pipeline parquet_to_minio load step finished in 10.77 seconds
1 load package(s) were loaded to destination filesystem and into dataset taxis_parquet
The filesystem destination used s3://taxis location to store data
Load package 1788705960.7305224 is LOADED and contains no failed jobs
